In [18]:
import faiss

In [19]:
faiss.__version__

'1.16.0'

In [20]:
data = """
Helping Millions of Students Succeed
Sudhanshu's commitment to affordable education wasn't just a business strategy—it was his life's mission. 
Over the years, iNeuron has helped over 1.5 million students from 34+ countries, providing them with the skills they need to succeed in today's competitive job market. 
Many of these students, like Sudhanshu himself, came from disadvantaged backgrounds. 
They saw iNeuron as a lifeline—an opportunity to rise above their circumstances.

In 2022, iNeuron was acquired by PhysicsWallah in a deal worth ₹250 crore. 
While this acquisition was a significant milestone, Sudhanshu remained focused on his mission. 
Even after the acquisition, iNeuron continued to offer some of the most affordable and accessible tech courses in the world.

Sudhanshu's journey isn't just one of entrepreneurial success; it's also a story of dedication to teaching. 
Throughout his career, he has remained a passionate educator, constantly looking for ways to empower others through knowledge. 
Whether teaching courses in Big Data, Data Science, or programming, Sudhanshu has always sought to make complex subjects accessible to learners at all levels.

His commitment to affordable education has earned him the respect and admiration of countless students. 
Many credit Sudhanshu with changing their lives, helping them secure jobs, improve their skills, and break free from the limitations of their backgrounds.
"""

In [21]:
def chunk_text(text, chunk_size=50, overlap=10):
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

In [22]:
chunked_data = chunk_text(data)
chunked_data

["Helping Millions of Students Succeed Sudhanshu's commitment to affordable education wasn't just a business strategy—it was his life's mission. Over the years, iNeuron has helped over 1.5 million students from 34+ countries, providing them with the skills they need to succeed in today's competitive job market. Many of these students,",
 "succeed in today's competitive job market. Many of these students, like Sudhanshu himself, came from disadvantaged backgrounds. They saw iNeuron as a lifeline—an opportunity to rise above their circumstances. In 2022, iNeuron was acquired by PhysicsWallah in a deal worth ₹250 crore. While this acquisition was a significant milestone, Sudhanshu",
 "₹250 crore. While this acquisition was a significant milestone, Sudhanshu remained focused on his mission. Even after the acquisition, iNeuron continued to offer some of the most affordable and accessible tech courses in the world. Sudhanshu's journey isn't just one of entrepreneurial success; it's also a st

In [23]:
import os

from dotenv import load_dotenv
from openai import OpenAI
load_dotenv()
client = OpenAI(
    api_key=os.environ["EURI_API_KEY"],
    base_url="https://api.euron.one/api/v1/euri",
)

resp = client.embeddings.create(
    model="text-embedding-3-small",
    input=chunked_data,
)
print(f"Dimensions: {len(resp.data[0].embedding)}")
print(f"First 5 values: {resp.data[0].embedding[:5]}")

Dimensions: 1536
First 5 values: [0.0018072128295898438, -0.0086822509765625, 0.067626953125, 0.0244140625, 0.032623291015625]


In [24]:
len(chunked_data)

6

In [25]:
len(resp.data)

6

In [26]:
type(resp.data)

list

In [27]:
import numpy as np
embeddings = np.array([item.embedding for item in resp.data])
print(f"Embeddings shape: {embeddings.shape}")

Embeddings shape: (6, 1536)


In [28]:
faiss_index = faiss.IndexFlatL2(embeddings.shape[1])
faiss_index.add(embeddings)
print(f"Number of vectors in the index: {faiss_index.ntotal}")

Number of vectors in the index: 6


In [29]:
faiss.write_index(faiss_index, "faiss_index.faiss")

In [30]:
query = "tell me about Sudhanshu's journey"
query_embedding = np.array([client.embeddings.create(
    model="text-embedding-3-small",
    input=query
).data[0].embedding])
D, I = faiss_index.search(query_embedding, k=5)  # Retrieve top 5 similar vectors
print(f"Distances: {D}")
print(f"Indices: {I}")

Distances: [[0.90617144 0.9486141  0.9669054  1.0650195  1.0980551 ]]
Indices: [[4 3 2 1 0]]


In [31]:
chunked_data[4]

'has always sought to make complex subjects accessible to learners at all levels. His commitment to affordable education has earned him the respect and admiration of countless students. Many credit Sudhanshu with changing their lives, helping them secure jobs, improve their skills, and break free from the limitations of their'

In [16]:
chunked_data[12]

'him the respect and admiration of countless students. Many credit Sudhanshu with changing their lives, helping them secure jobs, improve'

In [17]:
chunked_data[10]

'teaching courses in Big Data, Data Science, or programming, Sudhanshu has always sought to make complex subjects accessible to learners'